# InSilicoTrial MAS - interactive verification notebook

This notebook walks through the five verification steps of the technical specification:

1. **Initialization & environment checklist** - decide how to scale.
2. **Synthetic trial cohort** - build the digital-twin population.
3. **Distributed multi-agent batches** - run Protocol / Patient / Biostatistician agents.
4. **Delta Lake time travel** - audit trailing and point-in-time reads.
5. **Data lineage & MLflow tracing** - prove where every number came from.

It runs unchanged on **AWS Databricks (multi-node)**, **Databricks Community Edition
(single-node driver)** and a **laptop** (offline LLM provider, no credentials needed).

> All patients are synthetic. This is a trial-design decision-support tool, not clinical evidence.

## Cell 1 - Initialization & environment checklist

Checks PySpark/JVM availability, worker-python consistency, cluster shape and memory, then
recommends an execution backend: `spark` on a cluster, `local` (driver multiprocessing) on
Community Edition, `sequential` for tiny runs.

In [ ]:
# Install the platform in the notebook session (Databricks) or use the local venv.
# %pip install -e ..  # uncomment for an editable install from the repository root

# `display` exists on Databricks; outside it, fall back to print so the notebook still runs.
try:
    display  # noqa: B018
except NameError:
    def display(obj):  # type: ignore[misc]
        print(obj)

from insilico_trial_mas.config import load_config
from insilico_trial_mas.engine.checklist import inspect_environment

# Widgets are optional: outside Databricks the defaults below are used.
def widget(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)  # noqa: F821 - Databricks injects dbutils
        return dbutils.widgets.get(name)  # noqa: F821
    except Exception:
        return default

PROFILE = widget("config", "conf/simulation_community_edition.yaml")
N_PATIENTS = int(widget("n_patients", "10000"))

config = load_config(PROFILE, {"n_patients": N_PATIENTS})
report = inspect_environment(config)
print(report.render())

# On a cluster this prints the Spark master and executor count; on Community Edition it
# explains the fallback to driver-side multiprocessing.

## Cell 2 - Generation of the synthetic trial cohort

10,000 digital twins with demographics, comorbidity history, concomitant medication and
illustrative pharmacogenomic markers sampled from packaged population priors. Screening and
stratified block randomisation are performed by the **Protocol Agent**, which also records
every screen failure for the CONSORT diagram.

In [ ]:
from insilico_trial_mas.agents.base import AgentContext
from insilico_trial_mas.agents.protocol_agent import ProtocolAgent
from insilico_trial_mas.cohort.generator import CohortGenerator, cohort_summary
from insilico_trial_mas.cohort.serialization import profiles_to_frame
from insilico_trial_mas.pipeline import load_protocol

protocol = load_protocol(config.protocol_path)
cohort = CohortGenerator(protocol, config).generate()

protocol_agent = ProtocolAgent(
    AgentContext(
        run_id="NOTEBOOK-DRYRUN",
        protocol=protocol,
        config=config,
        seed=config.seed,
        total_epochs=config.epochs or protocol.epochs,
    )
)
screening, allocation = protocol_agent.plan(cohort)
cohort_frame = profiles_to_frame(screening.enrolled, allocation.arm_by_patient)

print(f"screened={screening.n_screened} enrolled={len(screening.enrolled)} "
      f"failures={screening.screen_failure_rate:.1%}")
print(f"allocation={allocation.arm_counts()}")
print(cohort_summary(screening.enrolled))

try:
    from pyspark.sql import SparkSession

    spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
    sdf = spark.createDataFrame(cohort_frame)
    sdf.createOrReplaceTempView("synthetic_cohort")
    display(sdf.groupBy("arm_id", "sex").count().orderBy("arm_id", "sex"))  # noqa: F821 - Databricks display
except Exception as exc:  # pandas-only environment
    print(f"pandas preview (no Spark session: {type(exc).__name__}):")
    print(cohort_frame.groupby(["arm_id", "sex"]).size())

## Cell 3 - Executing distributed multi-agent batches

Runs the full pipeline: PK/PD + learned ML physiology + LLM personas + biostatistician
analysis. On a cluster this is a Spark `applyInPandas` job (watch the **Spark Jobs** UI); on
Community Edition or a laptop the identical agent code runs through driver-side
multiprocessing. Both paths write the same Silver rows - verified by
`tests/test_engines.py::test_spark_engine_matches_sequential`.

In [ ]:
from insilico_trial_mas.pipeline import TrialSimulationPipeline

pipeline = TrialSimulationPipeline(config, protocol)
result = pipeline.run()

print(f"run id      : {result.run_id}")
print(f"engine      : {result.engine_result.backend}")
print(f"patients    : {result.engine_result.n_patients}")
print(f"silver      : {result.silver_location} (version {result.silver_version})")
print(f"adverse ev. : {len(result.analysis.adverse_events)}")
print(f"alerts      : {len(result.analysis.safety_alerts)}")

from insilico_trial_mas.agents.biostatistician_agent import observations_frame
observations = pipeline.store.read(
    pipeline.namespace.logical("silver", "patient_states"), run_id=result.run_id
)
observations[["patient_id", "arm_id", "epoch", "dose_mg", "plasma_conc_mg_l", "sbp", "sbp_change", "worst_ctcae_grade"]].head(8)

### Aggregates

The Biostatistician Agent's Gold tables: per-arm summaries, treatment-vs-control comparisons
with confidence intervals, adverse events by CTCAE grade and the DSMB stopping-rule log.

In [ ]:
import dataclasses
import pandas as pd

arms = pd.DataFrame([dataclasses.asdict(s) for s in result.analysis.arm_summaries])
comparisons = pd.DataFrame([dataclasses.asdict(c) for c in result.analysis.comparisons])
triggered = [e for e in result.analysis.stopping_evaluations if e.triggered]

print("Per-arm summary")
display(arms[["arm_id", "n_patients", "mean_sbp_change", "sd_sbp_change", "responder_rate", "ae_rate", "grade3_plus_rate", "sae_rate"]])  # noqa: F821
print("Comparisons vs control")
display(comparisons[["endpoint", "arm_id", "effect_estimate", "ci_low", "ci_high", "p_value", "q_value", "test"]])  # noqa: F821
print(f"stopping-rule evaluations triggered: {len(triggered)}")

## Cell 4 - Delta Lake time travel and audit trailing

Every write creates a new table version, so any historical readout can be reproduced exactly.
On Databricks with `storage.backend: delta` the SQL below is the real thing; the Python block
works on every backend (the local store implements the same version semantics).

In [ ]:
# On a Databricks cluster with Unity Catalog:
#
#   DESCRIBE HISTORY trial_simulations_prod.silver.patient_states;
#
#   SELECT arm_id,
#          count(*) AS observations,
#          approx_percentile(sbp_change, 0.5) AS median_sbp_change
#   FROM trial_simulations_prod.silver.patient_states VERSION AS OF 1
#   WHERE simulated_blood_pressure > 140   -- legacy column name; use `sbp` on current schemas
#   GROUP BY arm_id;
#
#   RESTORE TABLE trial_simulations_prod.silver.patient_states VERSION AS OF 2;

# Backend-independent equivalent (identical version semantics):
store = pipeline.store
table = pipeline.namespace.logical("silver", "patient_states")

history = pd.DataFrame(store.history(table, limit=10))
display(history[["version", "timestamp", "operation", "rows", "run_id"]])  # noqa: F821

high_bp_version_1 = store.read(table, version=1).query("sbp > 140")
print(f"version 1 rows with systolic BP > 140 mmHg: {len(high_bp_version_1)}")
high_bp_latest = store.read(table).query("sbp > 140")
print(f"latest version rows with systolic BP > 140 mmHg: {len(high_bp_latest)}")

# Replay proof: the run manifest records the seed, protocol digest, model digest and LLM policy.
import json
manifest = json.load(open(f"{result.output_dir}/run_manifest.json"))
print(json.dumps(manifest["replay"], indent=2))

## Cell 5 - Visualizing data lineage and MLflow tracing

Two complementary views:

* **Lineage** - which agent produced which table, from raw cohort to report (also written to
  `gold.lineage_edges`, complementing Unity Catalog's automatic table lineage).
* **MLflow tracing** - token distribution, request latency percentiles and the span hierarchy
  of each persona conversation.

In [ ]:
import json

print("Lineage summary:", result.lineage)
print()
print(result.lineage_mermaid)

# Which tables/artefacts did the Patient Persona agents feed?
graph = manifest["lineage_graph"]
edges = pd.DataFrame(graph["edges"])[["source", "target", "relation", "agent"]]
display(edges)  # noqa: F821

# MLflow: parameters, metrics and the report artefact of this run.
#   import mlflow
#   mlflow.search_runs(experiment_names=[config.tracking.experiment_name]).head()

In [ ]:
from insilico_trial_mas.mlflow_tracking.tracing import TraceStore

trace_path = config.tracking.trace_path
traces = TraceStore(trace_path)
summary = traces.summary(run_id=result.run_id)
print("LLM tracing summary (token distribution, latency percentiles):")
print(json.dumps(summary, indent=2))

trace_ids = traces.trace_ids(limit=1)
if trace_ids:
    tree = traces.tree(trace_ids[0])
    print(f"\nSpan hierarchy for trace {trace_ids[0]}:")
    def render(node, depth=0):
        tokens = f" tokens={node['tokens_in']}/{node['tokens_out']}" if node["tokens_in"] else ""
        print("  " * depth + f"- {node['name']} [{node['kind']}] {node['duration_ms']:.1f} ms{tokens}")
        for child in node["children"]:
            render(child, depth + 1)
    for root in tree["roots"]:
        render(root)
else:
    print("No persona calls were traced in this run (llm_mode may be 'off', or no trigger fired).")

## Where to go next

* `docs/ARCHITECTURE.md` - how the agents, engines and storage layers fit together.
* `docs/RUNBOOK_DATABRICKS_AWS.md` - cluster sizing, cost control, failure triage.
* `docs/SPEC_COMPLIANCE.md` - specification requirement -> implementation -> test map.
* `insilico-trial simulate --help` - the same pipeline from the command line or a job task.